# Week 2 — Data Collection, Cleaning and Preprocessing for Logistics Analysis

This notebook creates a reproducible preparation pipeline using the nine Olist CSV files. It covers data collection simulation, profiling, missing values, duplicates, relationship checks, feature engineering, outlier flags and scaling.

**Setup:** Put the nine CSV files in `data/raw`; save this notebook in `notebooks`. Install with `python -m pip install pandas numpy scikit-learn matplotlib seaborn`. Run every cell in order. The original raw files are never overwritten.

In [2]:
%pip install pandas numpy scikit-learn matplotlib seaborn

  Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached matplotlib-3.11.2-cp314-cp314-win_amd64.whl (9.5 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.4.0-cp314-cp314-win_amd64.whl (240 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p/"data"/"raw").is_dir(): return p
    raise FileNotFoundError("Create data/raw and place this notebook inside your project.")
ROOT=find_root()
RAW,OUT,REPORTS=ROOT/"data"/"raw",ROOT/"data"/"processed",ROOT/"reports"
OUT.mkdir(parents=True,exist_ok=True); REPORTS.mkdir(parents=True,exist_ok=True)
FILES={"customers":"olist_customers_dataset.csv","geolocation":"olist_geolocation_dataset.csv",
"orders":"olist_orders_dataset.csv","items":"olist_order_items_dataset.csv",
"payments":"olist_order_payments_dataset.csv","reviews":"olist_order_reviews_dataset.csv",
"products":"olist_products_dataset.csv","sellers":"olist_sellers_dataset.csv",
"translation":"product_category_name_translation.csv"}
tables={}
for name,file in FILES.items():
    if not (RAW/file).exists(): raise FileNotFoundError(RAW/file)
    tables[name]=pd.read_csv(RAW/file)
    print(name,tables[name].shape)

customers (99441, 5)
geolocation (1000163, 5)
orders (99441, 8)
items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
products (32951, 9)
sellers (3095, 4)
translation (71, 2)


## 1. Simulated collection and dataset structure

In a real logistics company, these records may come from an order management system, payment gateway, seller portal, warehouse system and carrier platform. For this assignment, the collection step is simulated by loading the publicly available Olist CSV extracts.

The tables have different grains. Orders are normally one row per order; order items can have several rows per order. This distinction matters because an incorrect join can multiply order values.

In [4]:
profile=[]; missing=[]
for name,df in tables.items():
    profile.append({"table":name,"rows":len(df),"columns":len(df.columns),
      "exact_duplicates":int(df.duplicated().sum()),"missing_cells":int(df.isna().sum().sum()),
      "missing_pct":round(df.isna().sum().sum()/max(df.size,1)*100,2),
      "memory_MB":round(df.memory_usage(deep=True).sum()/1024**2,2)})
    for col in df:
        missing.append({"table":name,"column":col,"dtype":str(df[col].dtype),
          "missing_count":int(df[col].isna().sum()),"missing_pct":round(df[col].isna().mean()*100,2)})
profile=pd.DataFrame(profile); missing=pd.DataFrame(missing).sort_values("missing_pct",ascending=False)
display(profile); display(missing.head(30))
profile.to_csv(REPORTS/"week2_profile.csv",index=False)
missing.to_csv(REPORTS/"week2_missing_audit.csv",index=False)

,table,rows,columns,exact_duplicates,missing_cells,missing_pct,memory_MB
0,customers,99441,5,0,0,0.00,26.59
1,geolocation,1000163,5,261831,0,0.00,130.26
2,orders,99441,8,0,4908,0.62,52.94
3,items,112650,7,0,0,0.00,35.99
4,payments,103886,5,0,0,0.00,16.23
5,reviews,99224,7,0,145903,21.01,39.12
6,products,32951,9,0,2448,0.83,6.30
7,sellers,3095,4,0,0,0.00,0.59
8,translation,71,2,0,0,0.00,0.01


,table,column,dtype,missing_count,missing_pct
33,reviews,review_comment_title,str,87656,88.34
34,reviews,review_comment_message,str,58247,58.70
16,orders,order_delivered_customer_date,str,2965,2.98
38,products,product_category_name,str,610,1.85
39,products,product_name_lenght,float64,610,1.85
40,products,product_description_lenght,float64,610,1.85
41,products,product_photos_qty,float64,610,1.85
15,orders,order_delivered_carrier_date,str,1783,1.79
14,orders,order_approved_at,str,160,0.16
43,products,product_length_cm,float64,2,0.01


## 2. Data quality and relational checks

Exact duplicates repeat every field. Repeated `order_id` in order-items is not necessarily a duplicate because one order can contain several items. Parent keys should be unique, while child foreign keys should match a parent record.

In [5]:
checks={
"orders.order_id unique":tables["orders"].order_id.is_unique,
"customers.customer_id unique":tables["customers"].customer_id.is_unique,
"products.product_id unique":tables["products"].product_id.is_unique,
"sellers.seller_id unique":tables["sellers"].seller_id.is_unique,
"items.order_id exists":tables["items"].order_id.isin(tables["orders"].order_id).all(),
"items.product_id exists":tables["items"].product_id.isin(tables["products"].product_id).all(),
"items.seller_id exists":tables["items"].seller_id.isin(tables["sellers"].seller_id).all()}
display(pd.DataFrame({"check":checks.keys(),"passed":checks.values()}))

,check,passed
0,orders.order_id unique,True
1,customers.customer_id unique,True
2,products.product_id unique,True
3,sellers.seller_id unique,True
4,items.order_id exists,True
5,items.product_id exists,True
6,items.seller_id exists,True


## 3. Clean copies and correct data types

The raw files remain untouched. We remove exact duplicate rows from working copies, trim whitespace from text, parse timestamps and convert known numeric columns. Invalid date strings become `NaT` so they can be audited instead of silently misread.

In [6]:
clean={}
for name,df in tables.items():
    x=df.drop_duplicates().copy()
    for col in x.select_dtypes(include="object"): x[col]=x[col].str.strip()
    clean[name]=x
date_cols=["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
"order_delivered_customer_date","order_estimated_delivery_date"]
for col in date_cols: clean["orders"][col]=pd.to_datetime(clean["orders"][col],errors="coerce")
for col in ["price","freight_value"]:
    clean["items"][col]=pd.to_numeric(clean["items"][col],errors="coerce")
orders=clean["orders"].copy(); products=clean["products"].copy()
print("Orders before:",len(tables["orders"]),"after exact duplicate removal:",len(orders))

C:\Users\SenthamizhSelvan\AppData\Local\Temp\ipykernel_27796\1678885007.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in x.select_dtypes(include="object"): x[col]=x[col].str.strip()
C:\Users\SenthamizhSelvan\AppData\Local\Temp\ipykernel_27796\1678885007.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

Orders before: 99441 after exact duplicate removal: 99441


## 4. Missing values — choose the method based on meaning

A missing actual delivery timestamp may mean that an order has not been delivered or that the event was not recorded. Median-imputing this field would invent a delivery event and distort service metrics. We retain it and add a flag.

For descriptive product measurements such as weight and dimensions, median imputation is a practical baseline because the median is less sensitive to extreme values than the mean. We also add a missingness indicator. Missing product categories are labelled `unknown` rather than guessed.

In [7]:
orders["delivery_timestamp_missing"]=orders["order_delivered_customer_date"].isna().astype("int8")
num_product=[c for c in ["product_name_lenght","product_description_lenght","product_photos_qty",
"product_weight_g","product_length_cm","product_height_cm","product_width_cm"] if c in products]
for col in num_product:
    products[col+"_was_missing"]=products[col].isna().astype("int8")
    products[col]=products[col].fillna(products[col].median())
if "product_category_name" in products:
    products["product_category_name"]=products["product_category_name"].fillna("unknown")
print("Missing delivery timestamps retained:",orders.delivery_timestamp_missing.sum())
print("Missing values in selected product fields:",products[num_product].isna().sum().sum())

Missing delivery timestamps retained: 2965
Missing values in selected product fields: 0


## 5. Feature engineering

A feature is a variable derived from source data to represent a business concept.

`actual_delivery_days = (customer delivery timestamp - purchase timestamp) / 86400`

`delivery_delay_days = customer delivery timestamp - estimated delivery timestamp`

A positive delay means late; zero or negative means on time/early. If the actual delivery timestamp is absent, the outcome remains unknown.

In [8]:
orders["actual_delivery_days"]=(orders.order_delivered_customer_date-orders.order_purchase_timestamp).dt.total_seconds()/86400
orders["estimated_delivery_days"]=(orders.order_estimated_delivery_date-orders.order_purchase_timestamp).dt.total_seconds()/86400
orders["delivery_delay_days"]=(orders.order_delivered_customer_date-orders.order_estimated_delivery_date).dt.total_seconds()/86400
valid=orders.order_delivered_customer_date.notna() & orders.order_estimated_delivery_date.notna()
orders["delivery_result"]=np.select(
 [valid & (orders.delivery_delay_days<=0),valid & (orders.delivery_delay_days>0)],
 ["On time","Late"],default="Unknown")
orders["purchase_year"]=orders.order_purchase_timestamp.dt.year
orders["purchase_month"]=orders.order_purchase_timestamp.dt.to_period("M").astype("string")
display(orders.delivery_result.value_counts(dropna=False))

delivery_result
On time    88649
Late        7827
Unknown     2965
Name: count, dtype: int64

## 6. Outlier detection using IQR

`IQR = Q3 - Q1`. Potential outliers lie below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR`. This is a screening method, not proof of an error. A very long delivery may be a real service failure, so this notebook flags outliers instead of automatically deleting them.

In [9]:
outlier_rows=[]
for col in ["actual_delivery_days","estimated_delivery_days","delivery_delay_days"]:
    s=orders[col].dropna()
    if len(s):
        q1,q3=s.quantile([.25,.75]); iqr=q3-q1
        lo,hi=q1-1.5*iqr,q3+1.5*iqr
        flag=orders[col].notna() & ((orders[col]<lo)|(orders[col]>hi))
        orders[col+"_iqr_flag"]=flag.astype("int8")
        outlier_rows.append({"feature":col,"lower_fence":lo,"upper_fence":hi,
          "outlier_count":int(flag.sum()),"observed":len(s)})
outliers=pd.DataFrame(outlier_rows); display(outliers)
outliers.to_csv(REPORTS/"week2_outliers.csv",index=False)

,feature,lower_fence,upper_fence,outlier_count,observed
0,actual_delivery_days,-6.664482,29.151212,4899,96476
1,estimated_delivery_days,3.191933,43.564618,2552,99441
2,delivery_delay_days,-31.025959,8.391576,4875,96476


## 7. Normalization and standardization

Min-Max scaling maps a feature to a chosen range (usually 0–1). Standardization expresses values as z-scores, with mean 0 and standard deviation 1. Scaling helps algorithms based on distance or gradient optimization; tree models often do not require it.

We keep original units for business interpretation. For predictive modelling, fit scalers on training data only to prevent data leakage.

In [10]:
scale_cols=["actual_delivery_days","estimated_delivery_days","delivery_delay_days"]
complete=orders[scale_cols].notna().all(axis=1)
if complete.any():
    mm=MinMaxScaler().fit_transform(orders.loc[complete,scale_cols])
    zz=StandardScaler().fit_transform(orders.loc[complete,scale_cols])
    for i,col in enumerate(scale_cols):
        orders.loc[complete,col+"_minmax"]=mm[:,i]
        orders.loc[complete,col+"_zscore"]=zz[:,i]
display(orders[scale_cols].describe())

,actual_delivery_days,estimated_delivery_days,delivery_delay_days
count,96476.000000,99441.000000,96476.000000
mean,12.558702,23.767650,-11.179120
std,9.546530,8.832371,10.186113
min,0.533414,1.648993,-146.016123
25%,6.766403,18.331690,-16.244384
50%,10.217755,23.240370,-11.948941
75%,15.720327,28.424861,-6.390000
max,209.628611,155.135463,188.975081


## 8. Join tables at the correct grain

The analysis table is item-level: an order containing three products produces three rows. `validate='many_to_one'` confirms that each child row maps to no more than one parent row. This protects against accidental row multiplication.

In [11]:
items=clean["items"].merge(
 orders[["order_id","order_status","order_purchase_timestamp","actual_delivery_days",
         "delivery_delay_days","delivery_result"]],
 on="order_id",how="left",validate="many_to_one")
items=items.merge(products[["product_id","product_category_name"]+num_product],
 on="product_id",how="left",validate="many_to_one")
items=items.merge(clean["sellers"][["seller_id","seller_zip_code_prefix","seller_city","seller_state"]],
 on="seller_id",how="left",validate="many_to_one")
trans=clean["translation"]
if {"product_category_name","product_category_name_english"}.issubset(trans.columns):
    items=items.merge(trans,on="product_category_name",how="left",validate="many_to_one")
print("Item-level rows:",len(items)); display(items.head())

Item-level rows: 112650


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,order_status,order_purchase_timestamp,actual_delivery_days,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,delivered,2017-09-13 08:59:02,7.614421,...,598.0,4.0,650.0,28.0,9.0,14.0,27277,volta redonda,SP,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,delivered,2017-04-26 10:53:06,16.216181,...,239.0,2.0,30000.0,50.0,30.0,40.0,3471,sao paulo,SP,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,delivered,2018-01-14 14:33:31,7.948437,...,695.0,2.0,3050.0,33.0,13.0,33.0,37564,borda da mata,MG,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,delivered,2018-08-08 10:00:35,6.147269,...,480.0,1.0,200.0,16.0,10.0,15.0,14403,franca,SP,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,delivered,2017-02-04 13:57:51,25.114352,...,409.0,1.0,3750.0,35.0,40.0,30.0,87900,loanda,PR,garden_tools


## 9. Save and verify outputs

Expected outputs include processed order, product and item tables plus audit CSVs. Check the file paths before proceeding to Week 3.

In [12]:
orders.to_csv(OUT/"orders_preprocessed.csv",index=False)
products.to_csv(OUT/"products_preprocessed.csv",index=False)
items.to_csv(OUT/"logistics_items_analysis.csv",index=False)
profile.to_csv(REPORTS/"week2_profile.csv",index=False)
for p in [OUT/"orders_preprocessed.csv",OUT/"products_preprocessed.csv",
          OUT/"logistics_items_analysis.csv",REPORTS/"week2_missing_audit.csv",
          REPORTS/"week2_outliers.csv"]:
    print(p.name,p.exists(),p.stat().st_size if p.exists() else "MISSING")

orders_preprocessed.csv True 36358992
products_preprocessed.csv True 3302040
logistics_items_analysis.csv True 33036689
week2_missing_audit.csv True 2070
week2_outliers.csv True 269


## 10. Reflection and submission evidence

After execution, use your generated audit files to answer: Which fields have the most missing values? Why were delivery timestamps not imputed? Which relationships passed validation? Which fields have potential outliers? How could an incorrect join distort freight totals?

For a strong submission, include actual counts and percentages from your run, screenshots of the audit outputs, and a short explanation of how each preprocessing decision affects later analysis. Do not invent results.